# Sportmonks Datenakquise — Matchstatistiken Super League

Ergänzt die API-Football Basisdaten um **detaillierte Matchstatistiken** (Sportmonks Football API v3).

**Zusätzliche Daten pro Spiel & Team:**  
🔵 Ballbesitz · 🎯 Schüsse · 🟨 Karten · 🚩 Eckbälle · 💢 Fouls · ↗️ Abseits · 🧤 Saves · ⚡ Angriffe · 🔁 Pässe

**Voraussetzung** — `.env` im Projekt-Root:
```
API_SPORTMONKS_KEY="dein_key"
API_SPORTMONKS_URL="https://api.sportmonks.com/v3/football"
```

**Output-Dateien in `data_acquisition/raw/`:**
| Datei | Inhalt | Verwendung |
|---|---|---|
| `teams_sportmonks.csv` | Team-IDs & Metadaten | Referenz |
| `team_season_stats.csv` | Aggregierte Saison-Totals pro Team | Radar Chart, Scatter Plot |
| `fixture_statistics.csv` | Spiel-für-Spiel Stats (eine Zeile pro Team pro Spiel) | Heatmap |
| `season_timeline.csv` | Kumulierte Punkte pro Spieltag | Liniendiagramm |
| `teams_combined.csv` | Sportmonks + API-Football gejoined | Scatter: Ballbesitz vs. Punkte |

---
**v3-Besonderheiten:**
- Scores: `description` (`CURRENT`/`1ST_HALF`/`2ND_HALF`) + `score.participant` (`home`/`away`)
- State: `include=state` → Objekt mit `short_name` (`FT`, `NS`, `AET`, …)
- Statistiken: `type_id`-basiert, nur erfasste Werte, via `location: home|away`
- Season ID 25607 = Swiss Super League 2024/2025 (via Postman-Collection bestätigt)

## 1. Setup & Imports

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("__file__").resolve().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

API_KEY  = os.environ["API_SPORTMONKS_KEY"]
BASE_URL = os.environ.get("API_SPORTMONKS_URL", "https://api.sportmonks.com/v3/football").rstrip("/")

if "docs.sportmonks" in BASE_URL:
    raise ValueError(
        "API_SPORTMONKS_URL zeigt auf die Doku, nicht die API!\n"
        "Korrekt: API_SPORTMONKS_URL=https://api.sportmonks.com/v3/football"
    )

HEADERS = {"Authorization": API_KEY}
RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

print(f"Base URL : {BASE_URL}")
print(f"API Key  : {'✓ geladen' if API_KEY else '✗ FEHLT'}")
print(f"Output   : {RAW_DIR.resolve()}")

## 2. Hilfsfunktionen

In [ ]:
def api_get(endpoint: str, params: dict = None) -> dict:
    """Einzelner GET-Request an Sportmonks v3."""
    url  = f"{BASE_URL}/{endpoint.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params or {})
    resp.raise_for_status()
    return resp.json()


def api_get_all(endpoint: str, params: dict = None) -> list:
    """Paginierter GET: lädt alle Seiten (via has_more) und gibt flache Liste zurück."""
    params = {**(params or {}), "per_page": 50}
    all_data, page = [], 1
    while True:
        data      = api_get(endpoint, {**params, "page": page})
        items     = data.get("data", [])
        all_data.extend(items)
        rate_rem  = data.get("rate_limit", {}).get("remaining", "?")
        has_more  = data.get("pagination", {}).get("has_more", False)
        print(f"  Seite {page:>2} | +{len(items):>3} Einträge | Rate verbleibend: {rate_rem}")
        if not has_more:
            break
        page += 1
        time.sleep(0.4)
    return all_data


def parse_participants(participants: list) -> tuple[dict, dict]:
    """Gibt (home_team, away_team) zurück. Location steht in participant.meta.location."""
    home = {"team_id": None, "team_name": ""}
    away = {"team_id": None, "team_name": ""}
    for p in participants or []:
        loc   = (p.get("meta") or {}).get("location", "")
        entry = {"team_id": p["id"], "team_name": p.get("name", "")}
        if loc == "home":  home = entry
        elif loc == "away": away = entry
    return home, away


def parse_scores(scores: list) -> dict:
    """Extrahiert Tore für alle Abschnitte aus dem scores-Array.

    v3: Zwei Einträge pro description (1ST_HALF / 2ND_HALF / CURRENT),
    jeweils einer mit score.participant='home' und einer mit 'away'.
    CURRENT = Endstand bei abgeschlossenen Spielen.
    """
    result = {}
    for sc in scores or []:
        desc        = sc.get("description", "").upper()
        score_obj   = sc.get("score", {})
        participant = score_obj.get("participant", "")
        goals       = score_obj.get("goals")
        if desc in {"CURRENT", "1ST_HALF", "2ND_HALF"} and participant in ("home", "away"):
            result[f"score_{desc.lower()}_{participant}"] = goals
    result["goals_home"] = result.get("score_current_home")
    result["goals_away"] = result.get("score_current_away")
    return result


def parse_state(state_field) -> str:
    """Gibt short_name zurück (FT / NS / AET …). Fallback auf developer_name."""
    if isinstance(state_field, dict):
        return state_field.get("short_name", "") or state_field.get("developer_name", "")
    return ""


def parse_fixture_statistics(statistics: list, type_map: dict) -> tuple[dict, dict]:
    """Pivotiert type_id-basierte Fixture-Stats in (home_stats, away_stats).
    v3: Nur erfasste Werte; Felder via location: home|away.
    """
    home_stats, away_stats = {}, {}
    for stat in statistics or []:
        type_id  = stat.get("type_id")
        location = stat.get("location", "")
        value    = (stat.get("data") or {}).get("value")
        col_name = type_map.get(type_id, f"stat_type_{type_id}")
        if location == "home":  home_stats[col_name] = value
        elif location == "away": away_stats[col_name] = value
    return home_stats, away_stats


print("✓ Hilfsfunktionen bereit")

## 3. Statistik-Typ-Mapping (type_id → Spaltenname)

Bekannte IDs hardcoded. Unbekannte werden nach dem Fixtures-Laden via `GET /types/{id}` aufgelöst.

In [ ]:
def make_col_name(name: str) -> str:
    return (
        name.lower()
        .replace(" ", "_").replace("/", "_").replace("-", "_")
        .replace("(", "").replace(")", "").replace("%", "pct")
        .replace(".", "").replace("'", "")
    )


# Bekannte Sportmonks v3 Statistik-Typ-IDs.
# /v3/football/types Listing gibt 404 (Core-API-Pfad) → Einzellookups als Fallback.
TYPE_MAP: dict[int, str] = {
    34: "tackles",
    41: "shots_total",
    42: "shots_on_target",
    43: "shots_off_target",
    44: "shots_blocked",
    45: "ball_possession_pct",
    51: "ball_safe",
    52: "goals",
    53: "passes_pct",
    54: "corner_kicks",
    56: "fouls",
    57: "offsides",
    58: "saves",
    80: "passes_total",
    81: "passes_accurate",
    83: "red_cards",
    84: "yellow_cards",
    86: "dangerous_attacks",
    87: "attacks",
}


def discover_unknown_types(type_ids: set, existing_map: dict) -> dict:
    """Löst unbekannte type_ids via GET /types/{id} einzeln auf."""
    unknown  = sorted(tid for tid in type_ids if tid not in existing_map and tid is not None)
    enriched = dict(existing_map)
    if not unknown:
        print(f"  Alle {len(type_ids)} Type-IDs bereits bekannt ✓")
        return enriched
    print(f"  {len(unknown)} unbekannte Type-IDs → löse via GET /types/{{id}} auf...")
    for tid in unknown:
        try:
            obj  = (api_get(f"types/{tid}").get("data") or {})
            name = obj.get("name") or obj.get("developer_name") or f"stat_type_{tid}"
            enriched[tid] = make_col_name(name)
            print(f"    {tid:>4}: {enriched[tid]}")
        except Exception:
            enriched[tid] = f"stat_type_{tid}"
            print(f"    {tid:>4}: ❌ → stat_type_{tid}")
        time.sleep(0.2)
    return enriched


print(f"TYPE_MAP bereit: {len(TYPE_MAP)} bekannte Typen vorgeladen.")

## 4. Season ID & Teams — Swiss Super League 2024/2025

In [ ]:
# Season ID 25607 = Swiss Super League 2024/2025
# Bestätigt via Postman-Collection (https://postman.sportmonks.com)
SEASON_ID = 25607

print(f"Saison: Swiss Super League 2024/2025  (ID {SEASON_ID})\n")
try:
    s = api_get(f"seasons/{SEASON_ID}").get("data", {})
    print(f"✅ {s.get('name', '?')}  ({s.get('starting_at', '?')} → {s.get('ending_at', '?')})")
except Exception as e:
    print(f"⚠️  Saison-Check fehlgeschlagen: {e}")

# Alle Teams in der Saison → Team-IDs für Statistik-Requests
print(f"\nLade alle Teams der Saison {SEASON_ID}...")
teams_raw = api_get_all(f"teams/seasons/{SEASON_ID}")

df_teams_sm = pd.DataFrame([
    {
        "team_id":   t["id"],
        "team_name": t.get("name", ""),
        "short_name": t.get("short_name", ""),
        "team_code": t.get("code", ""),
        "founded":   t.get("founded"),
        "logo":      t.get("image_path", ""),
    }
    for t in teams_raw
])
df_teams_sm.to_csv(RAW_DIR / "teams_sportmonks.csv", index=False)
TEAM_IDS = df_teams_sm["team_id"].tolist()

print(f"\n✅ {len(df_teams_sm)} Teams gespeichert:")
print(df_teams_sm[["team_id", "team_name", "short_name"]].to_string(index=False))

## 5. Saison-Statistiken pro Team (direkte API-Totals)

**Endpoint:** `GET /statistics/seasons/teams/{team_id}?filters=seasonId:25607&include=details`  
Liefert aggregierte Saison-Totals direkt — kein manuelles Aggregieren nötig.  
Response: `details[]` mit `type_id` + `value`-Objekt (Struktur variiert je Stat-Typ).

In [ ]:
def parse_stat_value(value) -> float | None:
    """Extrahiert den relevantesten numerischen Wert aus dem value-Objekt.

    Mögliche Strukturen:
      {"total": 45}                    → total
      {"average": "2.50", "total": 25} → total bevorzugt
      {"goals": 3, "penalties": 1}     → goals
      {"highest": 90, "lowest": 10}    → highest
    """
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, dict):
        for key in ("total", "goals", "average", "highest", "value"):
            if key in value and value[key] is not None:
                try:
                    return float(value[key])
                except (TypeError, ValueError):
                    pass
    return None


def fetch_team_season_stats(team_id: int, season_id: int, type_map: dict) -> dict:
    """Holt aggregierte Saison-Stats für ein Team und gibt flaches Dict zurück."""
    try:
        resp = api_get(
            f"statistics/seasons/teams/{team_id}",
            params={"filters": f"seasonId:{season_id}", "include": "details"}
        )
    except Exception as e:
        print(f"    ⚠️  {e}")
        return {}

    data    = resp.get("data", [])
    records = data if isinstance(data, list) else [data]
    record  = next(
        (r for r in records if r.get("season_id") == season_id),
        records[0] if records else None
    )
    if not record:
        return {}

    return {
        type_map.get(d.get("type_id"), f"stat_type_{d.get('type_id')}"): parse_stat_value(d.get("value"))
        for d in record.get("details", [])
        if parse_stat_value(d.get("value")) is not None
    }


assert TEAM_IDS, "TEAM_IDS nicht gesetzt — Zelle 4 zuerst ausführen!"
print(f"Lade Saison-Statistiken für {len(TEAM_IDS)} Teams (Season {SEASON_ID})...\n")

team_season_rows = []
for team_id in TEAM_IDS:
    name  = df_teams_sm.loc[df_teams_sm["team_id"] == team_id, "team_name"].values[0]
    print(f"  → {name} (ID {team_id})")
    stats = fetch_team_season_stats(team_id, SEASON_ID, TYPE_MAP)
    team_season_rows.append({"team_id": team_id, "team_name": name, **stats})
    time.sleep(0.4)

df_team_season = pd.DataFrame(team_season_rows)
df_team_season.to_csv(RAW_DIR / "team_season_stats.csv", index=False)

stat_cols = [c for c in df_team_season.columns if c not in {"team_id", "team_name"}]
print(f"\n✅ team_season_stats.csv gespeichert")
print(f"   {len(df_team_season)} Teams × {len(df_team_season.columns)} Spalten")
print(f"   Spalten: {', '.join(stat_cols)}")
df_team_season

## 6. Fixtures mit Spiel-für-Spiel-Statistiken laden

**Endpoint:** `GET /fixtures/seasons/25607?include=statistics;participants;scores;state`

In [ ]:
print(f"Lade alle Fixtures für Season {SEASON_ID}...\n")
fixtures_raw = api_get_all(
    f"fixtures/seasons/{SEASON_ID}",
    params={"include": "statistics;participants;scores;state"}
)

n_total = len(fixtures_raw)
n_ft    = sum(1 for fx in fixtures_raw if parse_state(fx.get("state")) in ("FT", "AET", "PEN"))
print(f"\n{n_total} Fixtures geladen ({n_ft} abgeschlossen, {n_total - n_ft} ausstehend).")

# Alle type_ids aus Fixture-Statistiken sammeln → unbekannte auflösen
fixture_type_ids = {
    stat.get("type_id")
    for fx in fixtures_raw
    for stat in (fx.get("statistics") or [])
    if stat.get("type_id") is not None
}
print(f"\n{len(fixture_type_ids)} einzigartige type_ids: {sorted(fixture_type_ids)}")
TYPE_MAP = discover_unknown_types(fixture_type_ids, TYPE_MAP)
print(f"\nAktives TYPE_MAP ({len(TYPE_MAP)} Einträge):")
print("  " + ", ".join(f"{k}={v}" for k, v in sorted(TYPE_MAP.items()) if k in fixture_type_ids))

## 7. Fixtures flachklopfen → `fixture_statistics.csv`

Eine Zeile pro Team pro Spiel (Home-Perspektive + Away-Perspektive).

In [ ]:
FINISHED_STATES = {"FT", "AET", "PEN", "BREAK", "ET"}
home_rows, away_rows, skipped = [], [], 0

for fx in fixtures_raw:
    status = parse_state(fx.get("state"))
    if status not in FINISHED_STATES:
        skipped += 1
        continue

    home_team, away_team   = parse_participants(fx.get("participants", []))
    score_data             = parse_scores(fx.get("scores", []))
    home_stats, away_stats = parse_fixture_statistics(fx.get("statistics", []), TYPE_MAP)

    base = {
        "fixture_id":   fx["id"],
        "date":         fx.get("starting_at", ""),
        "round_id":     fx.get("round_id"),
        "status":       status,
        "home_team_id": home_team["team_id"],
        "home_team":    home_team["team_name"],
        "away_team_id": away_team["team_id"],
        "away_team":    away_team["team_name"],
        **score_data,
    }
    home_rows.append({**base, "perspective": "home",
                      "team_id":        home_team["team_id"],
                      "team_name":      home_team["team_name"],
                      "goals_scored":   score_data.get("goals_home"),
                      "goals_conceded": score_data.get("goals_away"),
                      **home_stats})
    away_rows.append({**base, "perspective": "away",
                      "team_id":        away_team["team_id"],
                      "team_name":      away_team["team_name"],
                      "goals_scored":   score_data.get("goals_away"),
                      "goals_conceded": score_data.get("goals_home"),
                      **away_stats})

df_fixture_stats = (
    pd.concat([pd.DataFrame(home_rows), pd.DataFrame(away_rows)], ignore_index=True)
    .sort_values(["fixture_id", "perspective"]).reset_index(drop=True)
)
df_fixture_stats.to_csv(RAW_DIR / "fixture_statistics.csv", index=False)

meta_cols = {
    "fixture_id", "date", "round_id", "status", "perspective",
    "home_team_id", "home_team", "away_team_id", "away_team",
    "team_id", "team_name", "goals_home", "goals_away",
    "goals_scored", "goals_conceded",
    "score_current_home", "score_current_away",
    "score_1st_half_home", "score_1st_half_away",
    "score_2nd_half_home", "score_2nd_half_away",
}
stat_cols = [c for c in df_fixture_stats.columns if c not in meta_cols]

print(f"✅ fixture_statistics.csv gespeichert")
print(f"   {len(df_fixture_stats) // 2} Spiele | {skipped} ausstehend")
print(f"   {len(df_fixture_stats.columns)} Spalten total, {len(stat_cols)} Statistik-Spalten")
print(f"   Stat-Spalten: {', '.join(stat_cols)}")
df_fixture_stats.head(4)

## 8. Saisonverlauf → `season_timeline.csv`

Kumulierte Punkte, Tore, Siege/Unentschieden/Niederlagen pro Spieltag — Grundlage für das **Liniendiagramm**.

In [ ]:
def match_points(gs, gc) -> int | None:
    if pd.isna(gs) or pd.isna(gc): return None
    gs, gc = int(gs), int(gc)
    return 3 if gs > gc else (1 if gs == gc else 0)


timeline_rows = [
    {
        "fixture_id":     row["fixture_id"],
        "date":           row["date"],
        "round_id":       row["round_id"],
        "team_id":        row["team_id"],
        "team_name":      row["team_name"],
        "perspective":    row["perspective"],
        "goals_scored":   row.get("goals_scored"),
        "goals_conceded": row.get("goals_conceded"),
        "points":         (pts := match_points(row.get("goals_scored"), row.get("goals_conceded"))),
        "win":            1 if pts == 3 else 0,
        "draw":           1 if pts == 1 else 0,
        "loss":           1 if pts == 0 else 0,
    }
    for _, row in df_fixture_stats.iterrows()
]

df_timeline = pd.DataFrame(timeline_rows)
df_timeline["date"] = pd.to_datetime(df_timeline["date"], utc=True)
df_timeline = df_timeline.sort_values(["team_name", "date"]).reset_index(drop=True)

for col in ("points", "goals_scored", "goals_conceded", "win", "draw", "loss"):
    df_timeline[f"{col}_cumulative"] = df_timeline.groupby("team_id")[col].cumsum()
df_timeline["match_nr"] = df_timeline.groupby("team_id").cumcount() + 1

df_timeline.to_csv(RAW_DIR / "season_timeline.csv", index=False)
print(f"✅ season_timeline.csv gespeichert  ({len(df_timeline)} Zeilen)")

thun = df_timeline[df_timeline["team_name"].str.contains("Thun", case=False, na=False)]
if not thun.empty:
    print(f"\nFC Thun — letzte 5 Spiele:")
    print(thun[["match_nr","date","perspective","goals_scored","goals_conceded",
                "points","points_cumulative"]].tail(5).to_string(index=False))
else:
    print(f"\nℹ️  'Thun' nicht gefunden. Teams: {df_timeline['team_name'].unique().tolist()}")

## 9. Kombinierter Datensatz → `teams_combined.csv`

Joined `team_season_stats.csv` (Sportmonks) mit `standings.csv` (API-Football).  
Grundlage für **Scatter Plot: Ballbesitz % vs. Punkte**.

In [ ]:
standings_path = RAW_DIR / "standings.csv"

if not standings_path.exists():
    print("ℹ️  standings.csv nicht gefunden.")
    print("   Zuerst fetch_data.ipynb (API-Football) ausführen.")
else:
    df_standings = pd.read_csv(standings_path)
    df_combined  = df_standings.merge(
        df_team_season, on="team_name", how="left",
        suffixes=("_apifootball", "_sportmonks")
    )
    df_combined.to_csv(RAW_DIR / "teams_combined.csv", index=False)

    print(f"✅ teams_combined.csv gespeichert  ({len(df_combined)} Teams × {len(df_combined.columns)} Spalten)")
    unmatched = df_combined[
        df_combined[[c for c in df_team_season.columns if c not in {"team_id", "team_name"}][0]].isna()
    ]["team_name"].tolist() if len(df_team_season.columns) > 2 else []
    if unmatched:
        print(f"⚠️  Nicht gematchte Teams: {unmatched}")
        print("   Teamnamen zwischen API-Football und Sportmonks prüfen.")
    else:
        print("   Alle Teams gematchet ✓")

    scatter_cols = ["rank", "team_name", "points"] + [
        c for c in df_combined.columns if "possession" in c
    ]
    if len(scatter_cols) > 3:
        print()
        print(df_combined[scatter_cols].sort_values("rank").to_string(index=False))

## 10. Übersicht aller gespeicherten Dateien

In [ ]:
print("Gespeicherte Dateien in data_acquisition/raw/:\n")
for f in sorted(RAW_DIR.glob("*.csv")):
    df   = pd.read_csv(f)
    size = f.stat().st_size / 1024
    print(f"  {f.name:<38} {len(df):>4} Zeilen × {len(df.columns):>3} Spalten  ({size:.1f} KB)")

print("\n🏁 Sportmonks Datenakquise abgeschlossen.")
print("   Nächster Schritt: uv run python eda/generate-data-profile.py")